# EX - Auto MPG regressioon

Töö eesmärk on luua regressioonimudel, mis prognoosib autode kütusekulu (mpg) nende tehniliste parameetrite põhjal. Kasutatakse klassikalist andmestikku [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) UCI Machine Learning repositooriumist, mis sisaldab 398 kirjet ajavahemikust 1970–1982.

Töö käigus läbime kogu tsükli: andmete laadimine ja puhastamine, uurimuslik analüüs koos visualiseeringutega, tunnuste valik, kahe mudeli (LinearRegression ja Ridge) treenimine ning nende võrdlemine RMSE ja R² mõõdikute alusel.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from math import sqrt


## 1. Andmete laadimine

Fail `auto-mpg.data` ei sisalda päiseid ja on eraldatud tühikute/tabulatsioonidega, seetõttu kasutame `sep=r'\s+'` ja määrame veergude nimed käsitsi. Parameeter `na_values='?'` võimaldab pandasel kohe tuvastada puuduvad väärtused veerus `horsepower` (6 kirjet sisaldavad `?` arvu asemel).

Konstruktsioon `if 'google.colab' in sys.modules` tagab ühilduvuse: Google Colabis laaditakse andmed Drive'ilt, lokaalselt aga praegusest kaustast.


In [ ]:
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower',
                'weight', 'acceleration', 'model_year', 'origin', 'car_name']

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/MyDrive/google_colab/auto-mpg.data'
else:
    path = 'auto-mpg.data'

raw_data = pd.read_csv(path, sep=r'\s+', names=column_names, na_values='?')
raw_data.head()


Andmestik sisaldab 398 autokirjet (1970–1982) 9 veeruga:

| Tunnus | Kirjeldus |
|---|---|
| **mpg** | Kütusekulu (miili galloni kohta) — **sihtmuutuja** |
| **cylinders** | Silindrite arv (4, 6, 8) |
| **displacement** | Mootori töömaht (kuuptollides) |
| **horsepower** | Mootori võimsus (hobujõudu); sisaldab 6 puuduvat väärtust (`?`) |
| **weight** | Auto mass (naelades) |
| **acceleration** | Kiirendusaeg 0–60 miili/h (sekundites) |
| **model_year** | Mudeli aasta (70–82) |
| **origin** | Päritolu: 1 = USA, 2 = Euroopa, 3 = Aasia |
| **car_name** | Automudeli nimetus (tekstiline) |

**Miks `mpg` on sihtmuutuja:** kütusekulu on auto ökonoomsuse põhinäitaja, mis sõltub otseselt tehnilistest parameetritest (mass, võimsus, mootori maht). Kõik ülejäänud arvulised tunnused kirjeldavad auto konstruktsiooni ja sobivad `mpg` ennustajateks.

`describe()` väljundist on näha, et keskmine kütusekulu on ~23.5 mpg, vahemik 9 kuni 46.6. Mass varieerub 1613 kuni 5140 naela, võimsus 46 kuni 230 hj.

In [ ]:
raw_data.describe(include='all')

## 2. Andmete eeltöötlus

Peamised andmekvaliteedi probleemid selles andmestikus:

1. **Puuduvad väärtused veerus `horsepower`** (6 rida 398-st). Sümbol `?` on laadimisel juba töödeldud (`na_values='?'`), väärtused muutusid `NaN`-iks. Täitmise strateegia — **mediaan**: see on erindite suhtes vastupidav, erinevalt keskmisest, ega nihuta jaotust. Ridade kustutamine on samuti lubatud (kaotus <2%), kuid mediaaniga täitmine säilitab rohkem andmeid.

2. **Tüübiteisendus.** Igaks juhuks teisendame kõik arvulised veerud `pd.to_numeric(..., errors='coerce')` abil, et tuvastada võimalikke peidetud tekstiväärtusi nagu `-` või `n/a`.

3. **Duplikaadid.** Kontrollime arvuliste veergude alusel (erinevatel autodel võib olla sama nimi, kuid identsed tehnilised parameetrid tähendavad duplikaati).


In [ ]:
data_clean = raw_data.copy()

for col in ['mpg', 'cylinders', 'displacement', 'horsepower',
            'weight', 'acceleration', 'model_year', 'origin']:
    data_clean[col] = pd.to_numeric(data_clean[col], errors='coerce')

print(f"Puuduvad väärtused enne töötlust:\n{data_clean.isnull().sum()}\n")

# horsepower: 6 puuduvat väärtust — asendame mediaaniga (erindite suhtes vastupidav)
hp_median = data_clean['horsepower'].median()
data_clean['horsepower'] = data_clean['horsepower'].fillna(hp_median)
print(f"Horsepower mediaan täitmiseks: {hp_median}")

# Duplikaatide eemaldamine (kõigi veergude alusel peale car_name)
before = len(data_clean)
data_clean = data_clean.drop_duplicates(subset=data_clean.columns.difference(['car_name']))
print(f"Eemaldatud duplikaate: {before - len(data_clean)}")

print(f"\nPuuduvad väärtused pärast töötlust:\n{data_clean.isnull().sum()}")
data_clean


**`car_name` eemaldamine:** veerg sisaldab unikaalseid tekstilisi mudelinimetusi (ligi 300 unikaalset väärtust). Seda ei saa lineaarses regressioonis otse kasutada ja one-hot encoding tekitaks liiga palju hõredaid tunnuseid vähese vaatluste arvu juures (398). Info tootjariigi kohta on osaliselt kodeeritud läbi `origin`.

**`origin` jääb arvulise tunnusena:** väärtused 1, 2, 3 omavad mõttekat järjekorda (USA → Euroopa → Aasia) ja korreleeruvad autode suuruse/võimsusega. Lineaarse regressiooni jaoks on selline kodeerimine vastuvõetav.

In [ ]:
clean_numeric = data_clean.drop(columns=['car_name'])
print(f"Suurus: {clean_numeric.shape}")
print(f"Veerud: {list(clean_numeric.columns)}")
clean_numeric.head()


## 3. Tunnuste uurimine ja valik

Seoste hindamiseks koostame kolme tüüpi visualiseeringut:

- **Korrelatsiooni heatmap** — näitab paarikaupa lineaarseid seoseid kõigi tunnuste vahel. Võimaldab ühe pilguga näha, millised tunnused on tugevalt seotud `mpg`-ga (potentsiaalsed ennustajad) ja omavahel (multikollineaarsus).
- **Hajuvusdiagrammid** — 4 kõige tugevamalt `mpg`-ga korreleeruva tunnuse jaoks (`weight`, `horsepower`, `displacement`, `model_year`). Näitavad seose kuju (lineaarne, mittelineaarne) ja erindite olemasolu.
- **`mpg` histogramm** — võimaldab hinnata sihtmuutuja jaotust (normaalsus, asümmeetria, erindid).

Pärast visualiseerimist valime tunnused mudeli jaoks. Peamine tähelepanek: `weight`, `displacement`, `horsepower` ja `cylinders` on omavahel tugevalt korreleeritud (r > 0.85), mis tekitab multikollineaarsust. Osa neist võiks eemaldada, kuid otsustame jätta kõik 7 arvulist tunnust ja usaldada multikollineaarsuse käsitlemise Ridge-regulariseerimisele — see võimaldab mitte kaotada informatsiooni.


In [ ]:
# --- Korrelatsioonimaatriks ---
fig, ax = plt.subplots(figsize=(10, 8))
corr = clean_numeric.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Korrelatsioonimaatriks')
plt.tight_layout()
plt.show()

print("\nTunnuste korrelatsioon mpg-ga:")
print(corr['mpg'].drop('mpg').sort_values())

# --- Peamiste tunnuste hajuvusdiagrammid ---
key_features = ['weight', 'horsepower', 'displacement', 'model_year']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, feat in zip(axes.ravel(), key_features):
    ax.scatter(clean_numeric[feat], clean_numeric['mpg'], alpha=0.5, edgecolors='k', linewidth=0.3)
    ax.set_xlabel(feat)
    ax.set_ylabel('mpg')
    ax.set_title(f'mpg vs {feat}')
plt.tight_layout()
plt.show()

# --- Sihtmuutuja jaotus ---
fig, ax = plt.subplots(figsize=(8, 4))
clean_numeric['mpg'].hist(bins=25, ax=ax, edgecolor='black')
ax.set_xlabel('mpg')
ax.set_ylabel('Sagedus')
ax.set_title('mpg jaotus')
plt.tight_layout()
plt.show()


In [ ]:
# Kõik arvulised tunnused (peale sihtmuutuja) — kasutame täiskomplekti,
# kuna Ridge-regulariseerimine tuleb multikollineaarsusega toime
selected_features = ['cylinders', 'displacement', 'horsepower',
                     'weight', 'acceleration', 'model_year', 'origin']
print(f"Valitud {len(selected_features)} tunnust: {selected_features}")
selected_features


## 4. Treening- ja testandmed

Jagame andmed vahekorras **80/20** (318 / 80 kirjet). See on standardne proportsioon: treeningvalim on piisavalt suur stabiilseks treenimiseks ja testvalim (~80 kirjet) — mõõdikute usaldusväärseks hindamiseks.

**`random_state=42`** fikseerib juhusliku segamise, tagades korratavuse: notebooki uuesti käivitamisel jääb jaotus samaks ja mõõdikud on võrreldavad. Väärtus 42 on konventsiooni järgi üldlevinud ega mõjuta jaotuse kvaliteeti.


In [ ]:
X = clean_numeric[selected_features]
y = clean_numeric['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treeningvalim: {X_train.shape[0]} kirjet")
print(f"Testvalim:     {X_test.shape[0]} kirjet")


## 5. Andmete skaleerimine

Rakendame skaleerimist nii **tunnustele** kui ka **sihtmuutujale**:

**Tunnused — StandardScaler** (z-normaliseerimine: keskmine → 0, standardhälve → 1):
- **Skaalade ühtlustamine.** Tunnused on erinevatel skaaladel: `weight` ~ 1600–5100, `cylinders` ~ 3–8, `model_year` ~ 70–82. Ilma skaleerimiseta karistaks Ridge-regressioon koefitsiente ebaühtlaselt.
- **Koefitsientide tõlgendatavus.** Pärast StandardScalerit näitab iga koefitsient, kui palju mpg muutub tunnuse ühe standardhälbe võrra muutudes.

**Sihtmuutuja — MinMaxScaler** (normaliseerimine vahemikku [0, 1]):
- `mpg` skaleerimine normaliseerib RMSE: absoluutväärtuste (3–4 mpg) asemel väljendatakse viga sihtmuutuja vahemiku osana, mis muudab mõõdiku erinevate ülesannete vahel võrreldavaks.
- Mudelid treenitakse normaliseeritud väärtusi ennustama, mis parandab arvulist stabiilsust.

**Oluline:** `fit_transform` rakendatakse ainult treeningvalimile ja `transform` testvalimile. See hoiab ära andmelekke (data leakage).

In [ ]:
feature_scaler = StandardScaler()
target_scaler = MinMaxScaler()

X_train = pd.DataFrame(
    feature_scaler.fit_transform(X_train),
    columns=selected_features,
    index=X_train.index
)
X_test = pd.DataFrame(
    feature_scaler.transform(X_test),
    columns=selected_features,
    index=X_test.index
)

y_train = pd.Series(
    target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel(),
    index=y_train.index
)
y_test = pd.Series(
    target_scaler.transform(y_test.values.reshape(-1, 1)).ravel(),
    index=y_test.index
)

print("Tunnused — keskmised pärast skaleerimist (train, ≈0):")
print(X_train.mean().round(6))
print(f"\ny_train pärast MinMaxScalerit: min={y_train.min():.3f}, max={y_train.max():.3f}")


## 6. Mudelite treenimine

Treenime kaks mudelit:

1. **LinearRegression (OLS)** — klassikaline vähimruutude meetod, ilma regulariseerimiseta. Toimib baasmudeli (baseline) võrdluspunktina. Lihtne tõlgendada: iga koefitsient näitab tunnuse panust prognoosi.

2. **Ridge (alpha=1.0)** — lineaarne regressioon L2-regulariseerimisega. Lisab kaotuse funktsioonile trahvi $\alpha \cdot \sum \beta_i^2$, mis „surub" koefitsiendid nulli poole. See on eriti kasulik multikollineaarsuse korral: selle asemel, et üks korreleeruvatest tunnustest (nt `weight`) saaks suure positiivse koefitsiendi ja teine (`displacement`) suure negatiivse, jaotab Ridge kaalu ühtlasemalt ja stabiilsemalt.

`alpha=1.0` — mõõdukas regulariseerimise tase (vaikeväärtus). Põhjalikumaks valikuks saab kasutada ristvalideerimist (`RidgeCV`).

Pärast treenimist väljastame mõlema mudeli koefitsiendid — see võimaldab näha, kuidas regulariseerimine jaotab kaalu korreleeruvate tunnuste vahel ümber.


In [ ]:
# Baasne lineaarne regressioon — lähtepunkt võrdluseks
model_1 = LinearRegression()
model_1.fit(X_train, y_train)

# Ridge-regressioon — L2-regulariseerimine multikollineaarsuse vastu
# (displacement, weight, horsepower, cylinders on omavahel tugevalt korreleeritud)
model_2 = Ridge(alpha=1.0)
model_2.fit(X_train, y_train)

print("LinearRegression koefitsiendid:")
for feat, coef in zip(selected_features, model_1.coef_):
    print(f"  {feat:15s} {coef:+.3f}")

print(f"\nRidge (alpha=1.0) koefitsiendid:")
for feat, coef in zip(selected_features, model_2.coef_):
    print(f"  {feat:15s} {coef:+.3f}")


## 7. Mudelite hindamine

Hindamiseks kasutame kahte mõõdikut:

- **RMSE (Root Mean Squared Error)** — vigade ruutkeskmise ruutjuur. Kuna sihtmuutuja on MinMaxScaleriga skaleeritud vahemikku [0, 1], väljendatakse RMSE selle vahemiku osana. RMSE ≈ 0.08 tähendab, et mudeli keskmine viga on ~8% mpg vahemikust.

- **R² (determinatsioonikordaja)** — sihtmuutuja dispersiooni osa, mida mudel seletab. R² = 0.85 tähendab, et mudel seletab 85% mpg varieeruvusest, ülejäänud 15% on seletamata müra või arvestamata tegurite mõju. Skaleerimine R²-le mõju ei avalda.

Mõlemad mõõdikud arvutatakse **testvalimil** (80 kirjet), mis ei osalenud treenimises. Nimekiri `models` sorditakse RMSE järgi kasvavalt, et parim mudel oleks esimene.


In [ ]:
def rmse(y_true, y_pred):
    return sqrt(mean_squared_error(y_true, y_pred))

y_pred_1 = model_1.predict(X_test)
y_pred_2 = model_2.predict(X_test)

rmse_1 = rmse(y_test, y_pred_1)
rmse_2 = rmse(y_test, y_pred_2)

r2_1 = r2_score(y_test, y_pred_1)
r2_2 = r2_score(y_test, y_pred_2)

print(f"{'Mudel':<25s} {'RMSE':>8s} {'R²':>8s}")
print("-" * 43)
print(f"{'LinearRegression':<25s} {rmse_1:8.3f} {r2_1:8.3f}")
print(f"{'Ridge (alpha=1.0)':<25s} {rmse_2:8.3f} {r2_2:8.3f}")

models = [(model_1, rmse_1), (model_2, rmse_2)]
models.sort(key=lambda x: x[1])

print(f"\nParim mudel RMSE järgi: {type(models[0][0]).__name__} (RMSE={models[0][1]:.3f})")
models


## 8. Kokkuvõte

Allpool on tehtud töö ülevaade: peamised tähelepanekud andmestiku kohta, olulisemad tunnused ja mudelite võrdlus.


Auto MPG andmestik sisaldab 398 autokirjet aastatest 1970–1982, millel on 8 arvulist tunnust ja üks tekstiline muutuja (mudeli nimetus). 398 reast ainult 6 sisaldasid puuduvaid väärtusi — kõik veerus `horsepower`, mis asendati mediaaniga (93.5 hj), säilitades valimi suuruse ilma jaotust moonutamata. Duplikaate andmestikus ei leitud.

Korrelatsioonianalüüs näitas, et kütusekulu (mpg) mõjutavad kõige enam auto mass (`weight`, korrelatsioon −0.83), mootori töömaht (`displacement`, −0.80), võimsus (`horsepower`, −0.77) ja silindrite arv (`cylinders`, −0.78). Need neli tunnust on ka omavahel tugevalt korreleeritud, mis viitab multikollineaarsusele. Mudeli aasta (`model_year`) korreleerub mpg-ga positiivselt (+0.58), peegeldades trendi säästlikumate autode suunas. Tunnus `origin` näitab samuti positiivset seost (+0.56) — Aasia ja Euroopa autod on keskmiselt säästlikumad kui Ameerika omad.

Modelleerimiseks valiti kõik 7 arvulist tunnust. Tunnused skaleeriti `StandardScaler` abil (z-normaliseerimine) ja sihtmuutuja `mpg` — `MinMaxScaler` abil (normaliseerimine vahemikku [0, 1]). Sihtmuutuja skaleerimine normaliseerib RMSE, väljendades viga mpg vahemiku osana, mis muudab mõõdiku võrreldavaks.

Treeniti kaks mudelit: tavaline lineaarne regressioon (LinearRegression) ja Ridge-regressioon parameetriga alpha=1.0. Mõlemad mudelid näitasid sarnaseid tulemusi: R² ≈ 0.85 ja RMSE ≈ 0.08 (skaleeritud andmetel). Suurima panuse annavad `weight` ja `model_year`, mis on kooskõlas korrelatsioonianalüüsiga. Ridge-regressioon stabiliseerib koefitsiente multikollineaarsuse korral, jaotades kaalu korreleeruvate tunnuste vahel ümber, kuid erinevus lõppmõõdikutes on minimaalne — 7 tunnuse ja 398 vaatluse juures ei ole ülesobitamine kriitiline.

Lõplik R² ≈ 0.85 tähendab, et mudelid seletavad umbes 85% kütusekulu dispersioonist. Täpsuse tõstmiseks saab proovida mittelineaarseid mudeleid (Random Forest, Gradient Boosting) või lisada tuletatud tunnuseid (nt võimsuse ja massi suhe).